In [1]:
!unzip Plantar.zip

Archive:  Plantar.zip
   creating: Plantar/
   creating: Plantar/test/
   creating: Plantar/test/flat foot/
  inflating: Plantar/test/flat foot/image_0032_png_jpg.rf.3aea00de50f455a32eaca1368f01161f.jpg  
  inflating: Plantar/test/flat foot/image_0032_png_jpg.rf.fdfa53086a2792f31b2dc58904de553c.jpg  
  inflating: Plantar/test/flat foot/image_0035_png_jpg.rf.b5e811d704b2cd376d3bd78b6cda9319.jpg  
  inflating: Plantar/test/flat foot/image_0041_png_jpg.rf.54c7b8917b43268ceb9c810112e3fd39.jpg  
  inflating: Plantar/test/flat foot/image_0059_png_jpg.rf.5615261a0eabde9f95d4f1a902e2de7a.jpg  
  inflating: Plantar/test/flat foot/image_0060_png_jpg.rf.7c97742c30c2b1e2dd7a8615582e33e4.jpg  
  inflating: Plantar/test/flat foot/image_0077_png_jpg.rf.d3fb007887a148ec2c3aef897cb923c5.jpg  
  inflating: Plantar/test/flat foot/image_0078_png_jpg.rf.6c244ac452960c984116fafab7eae6c5.jpg  
  inflating: Plantar/test/flat foot/image_0101_png_jpg.rf.0b7ed5eb5e8185bf29d4bb82f0a3f448.jpg  
  inflating: Planta

CLEANING

In [2]:
import os
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [3]:
PLANTAR_PATH = "/content/Plantar"

In [4]:
# Class Mapping
def class_mapping():

    plantar = Path(PLANTAR_PATH)

    class_mapping = {
        'flat foot': 0,
        'normal': 1,
        'over-arch': 2
    }

    print("\nClass Mapping:")
    for class_name, class_id in class_mapping.items():
        print(f"  {class_name} → {class_id}")

    total_images = 0

    for split in ['train', 'test']:
        print(f"\n{split.upper()}:")

        split_path = plantar / split
        if not split_path.exists():
            print(f"{split} folder not found!")
            continue

        for class_name in ['flat foot', 'normal', 'over-arch']:
            folder = split_path / class_name
            if folder.exists():
                images = [f for f in folder.iterdir() if f.suffix.lower() in ['.jpg', '.png', '.bmp', '.tif', '.jpeg']]
                print(f"  {class_name}: {len(images)} images")
                total_images += len(images)
            else:
                print(f"  {class_name}: Folder not found!")

    print(f"\nTotal images: {total_images}")

    return class_mapping

class_mapping()


Class Mapping:
  flat foot → 0
  normal → 1
  over-arch → 2

TRAIN:
  flat foot: 225 images
  normal: 680 images
  over-arch: 487 images

TEST:
  flat foot: 48 images
  normal: 168 images
  over-arch: 125 images

Total images: 1733


{'flat foot': 0, 'normal': 1, 'over-arch': 2}

In [5]:
# Consistancy Check
def data_consistency():

    plantar = Path(PLANTAR_PATH)

    # Track statistics
    resolutions = {}
    formats = {}
    color_modes = {}
    corrupted_files = []

    all_images = []

    # Collect all image paths
    for split in ['train', 'test']:
        for class_name in ['flat foot', 'normal', 'over-arch']:
            folder = plantar / split / class_name
            if folder.exists():
                images = [f for f in folder.iterdir() if f.suffix.lower() in ['.jpg', '.png', '.bmp', '.tif', '.jpeg']]
                all_images.extend(images)

    print(f"\nAnalyzing {len(all_images)} images...")

    for img_path in all_images:
        try:
            # Read image
            img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)

            if img is None:
                corrupted_files.append(img_path.name)
                print(f"Could not read: {img_path.name}")
                continue

            # Check resolution
            h, w = img.shape[:2]
            resolution = f"{w}x{h}"
            resolutions[resolution] = resolutions.get(resolution, 0) + 1

            # Check format
            file_format = img_path.suffix.lower()
            formats[file_format] = formats.get(file_format, 0) + 1

            # Check color mode
            if len(img.shape) == 2:
                color_mode = "Grayscale"
            elif len(img.shape) == 3 and img.shape[2] == 3:
                color_mode = "3 channels (BGR/RGB)"
            elif len(img.shape) == 3 and img.shape[2] == 4:
                color_mode = "4 channels (BGRA/RGBA)"
            else:
                color_mode = f"Unknown ({img.shape})"

            color_modes[color_mode] = color_modes.get(color_mode, 0) + 1

        except Exception as e:
            corrupted_files.append(img_path.name)
            print(f"Error processing {img_path.name}: {str(e)}")

    # Print results
    print("\nRESOLUTIONS:")
    for res, count in sorted(resolutions.items(), key=lambda x: x[1], reverse=True):
        print(f"  {res}: {count} images")

    if len(resolutions) == 1:
        print("All images have consistent resolution")
    else:
        print("Multiple resolutions found - need standardization")

    print("\nFILE FORMATS:")
    for fmt, count in formats.items():
        print(f"  {fmt}: {count} images")

    if len(formats) == 1:
        print("All images have consistent format")
    else:
        print("Multiple formats found")

    print("\nCOLOR MODES:")
    for mode, count in color_modes.items():
        print(f"  {mode}: {count} images")

    if len(corrupted_files) > 0:
        print(f"\nCORRUPTED FILES: {len(corrupted_files)}")
        for filename in corrupted_files[:10]:
            print(f"  - {filename}")
    else:
        print("\nNo corrupted files found")

    return resolutions, formats, color_modes, corrupted_files

data_consistency()


Analyzing 1733 images...

RESOLUTIONS:
  640x640: 1733 images
All images have consistent resolution

FILE FORMATS:
  .jpg: 1733 images
All images have consistent format

COLOR MODES:
  3 channels (BGR/RGB): 1733 images

No corrupted files found


({'640x640': 1733}, {'.jpg': 1733}, {'3 channels (BGR/RGB)': 1733}, [])

In [6]:
# pixel ranges
def pixel_value_ranges():

    plantar = Path(PLANTAR_PATH)

    all_mins = []
    all_maxs = []
    all_means = []

    # Collect all image paths
    all_images = []
    for split in ['train', 'test']:
        for class_name in ['flat foot', 'normal', 'over-arch']:
            folder = plantar / split / class_name
            if folder.exists():
                images = [f for f in folder.iterdir() if f.suffix.lower() in ['.jpg', '.png', '.bmp', '.tif', '.jpeg']]
                all_images.extend(images)

    print(f"\nAnalyzing pressure values in {len(all_images)} images...")

    for img_path in all_images:
        try:
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                all_mins.append(img.min())
                all_maxs.append(img.max())
                all_means.append(img.mean())
        except:
            continue

    all_mins = np.array(all_mins)
    all_maxs = np.array(all_maxs)
    all_means = np.array(all_means)

    print("\nPRESSURE STATISTICS (Pixel Values):")
    print(f"  Global Min: {all_mins.min()}")
    print(f"  Global Max: {all_maxs.max()}")
    print(f"\n  Average across all images:")
    print(f"    Min: {all_mins.mean():.2f} (±{all_mins.std():.2f})")
    print(f"    Max: {all_maxs.mean():.2f} (±{all_maxs.std():.2f})")
    print(f"    Mean: {all_means.mean():.2f} (±{all_means.std():.2f})")

    return all_mins, all_maxs, all_means

pixel_value_ranges()


Analyzing pressure values in 1733 images...

PRESSURE STATISTICS (Pixel Values):
  Global Min: 0
  Global Max: 255

  Average across all images:
    Min: 0.00 (±0.00)
    Max: 255.00 (±0.00)
    Mean: 59.15 (±2.26)


(array([0, 0, 0, ..., 0, 0, 0], dtype=uint8),
 array([255, 255, 255, ..., 255, 255, 255], dtype=uint8),
 array([60.3907373 , 57.20407471, 63.49849854, ..., 57.9097876 ,
        61.28287354, 61.92292725]))

In [7]:
# Detect Outliers

def detect_outliers():

    plantar = Path(PLANTAR_PATH)

    image_stats = []

    # Collect statistics for all images
    for split in ['train', 'test']:
        for class_name in ['flat foot', 'normal', 'over-arch']:
            folder = plantar / split / class_name
            if not folder.exists():
                continue

            for img_path in folder.iterdir():
                if img_path.suffix.lower() not in ['.jpg', '.png', '.bmp', '.tif', '.jpeg']:
                    continue

                try:
                    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
                    if img is not None:
                        image_stats.append({
                            'path': img_path,
                            'min': img.min(),
                            'max': img.max(),
                            'mean': img.mean(),
                            'std': img.std()
                        })
                except:
                    continue

    # Detect outliers using IQR method on max values
    maxs = np.array([s['max'] for s in image_stats])
    Q1 = np.percentile(maxs, 25)
    Q3 = np.percentile(maxs, 75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    print(f"\nOutlier detection (using IQR method on max values):")
    print(f"  Q1 (25th percentile): {Q1:.2f}")
    print(f"  Q3 (75th percentile): {Q3:.2f}")
    print(f"  IQR: {IQR:.2f}")
    print(f"  Lower bound: {lower_bound:.2f}")
    print(f"  Upper bound: {upper_bound:.2f}")

    outliers = [s for s in image_stats if s['max'] < lower_bound or s['max'] > upper_bound]

    print(f"\nFound {len(outliers)} outliers ({len(outliers)/len(image_stats)*100:.2f}%)")

    if len(outliers) > 0:
        print("\nOutlier images:")
        for out in outliers[:10]:
            print(f"  {out['path'].name}: max={out['max']:.2f}, mean={out['mean']:.2f}")

    return outliers

detect_outliers()


Outlier detection (using IQR method on max values):
  Q1 (25th percentile): 255.00
  Q3 (75th percentile): 255.00
  IQR: 0.00
  Lower bound: 255.00
  Upper bound: 255.00

Found 0 outliers (0.00%)


[]

In [10]:
# Class Balance
def class_balance():

    plantar = Path(PLANTAR_PATH)

    class_counts = {
        'train': {'flat foot': 0, 'normal': 0, 'over-arch': 0},
        'test': {'flat foot': 0, 'normal': 0, 'over-arch': 0}
    }

    for split in ['train', 'test']:
        for class_name in ['flat foot', 'normal', 'over-arch']:
            folder = plantar / split / class_name
            if folder.exists():
                images = [f for f in folder.iterdir() if f.suffix.lower() in ['.jpg', '.png', '.bmp', '.tif', '.jpeg']]
                class_counts[split][class_name] = len(images)

    print("\nClass distribution:")

    for split in ['train', 'test']:
        print(f"\n{split.upper()}:")
        total = sum(class_counts[split].values())
        for class_name, count in class_counts[split].items():
            percentage = (count / total * 100) if total > 0 else 0
            print(f"  {class_name}: {count} ({percentage:.1f}%)")

    # Check if balanced (within 10% of each other)
    train_counts = list(class_counts['train'].values())
    test_counts = list(class_counts['test'].values())

    train_balanced = max(train_counts) / min(train_counts) < 1.5 if min(train_counts) > 0 else False
    test_balanced = max(test_counts) / min(test_counts) < 1.5 if min(test_counts) > 0 else False

    if train_balanced and test_balanced:
        print("\nClasses are reasonably balanced")
    else:
        print("\nClass imbalance detected")

    return class_counts

class_balance()


Class distribution:

TRAIN:
  flat foot: 225 (16.2%)
  normal: 680 (48.9%)
  over-arch: 487 (35.0%)

TEST:
  flat foot: 48 (14.1%)
  normal: 168 (49.3%)
  over-arch: 125 (36.7%)

Class imbalance detected


{'train': {'flat foot': 225, 'normal': 680, 'over-arch': 487},
 'test': {'flat foot': 48, 'normal': 168, 'over-arch': 125}}

In [12]:
# FIX
import os
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

PLANTAR_PATH = "/content/Plantar"
OUTPUT_PATH = "/content/Plantar_C"
def clean_plantar_data():

    plantar = Path(PLANTAR_PATH)
    output = Path(OUTPUT_PATH)

    # Create output directory structure
    print("Creating output directory structure...")
    for split in ['train', 'test']:
        for class_name in ['flat foot', 'normal', 'over-arch']:
            (output / split / class_name).mkdir(parents=True, exist_ok=True)

    print(f"Output directory created: {output}\n")

    # Statistics
    stats = {
        'total': 0,
        'converted_to_grayscale': 0,
        'resized': 0,
        'already_correct': 0
    }

    # Process each image
    print("Processing images...\n")

    for split in ['train', 'test']:
        print(f"Processing {split.upper()} set...")

        for class_name in ['flat foot', 'normal', 'over-arch']:
            folder = plantar / split / class_name
            if not folder.exists():
                continue

            images = [f for f in folder.iterdir() if f.suffix.lower() in ['.jpg', '.png', '.jpeg']]

            for img_path in tqdm(images, desc=f"  {class_name}"):
                stats['total'] += 1

                # Read image
                img = cv2.imread(str(img_path))
                if img is None:
                    print(f"Could not read: {img_path.name}")
                    continue

                was_modified = False

                # Step 1: Convert to grayscale
                if len(img.shape) == 3:
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                    stats['converted_to_grayscale'] += 1
                    was_modified = True

                # Step 2: Resize to 224x224
                if img.shape != (224, 224):
                    img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_LINEAR)
                    stats['resized'] += 1
                    was_modified = True

                if not was_modified:
                    stats['already_correct'] += 1

                # Save cleaned image
                output_path = output / split / class_name / img_path.name
                cv2.imwrite(str(output_path), img)

    # Print statistics
    print("\n" + "=" * 60)
    print("CLEANING COMPLETE!")
    print("=" * 60)
    print(f"\nStatistics:")
    print(f"  Total images processed: {stats['total']}")
    print(f"  Converted to grayscale: {stats['converted_to_grayscale']}")
    print(f"  Resized to 224x224: {stats['resized']}")
    print(f"  Already correct: {stats['already_correct']}")
    print(f"\nCleaned data saved to: {output}")


def verify_cleaned_data():
    """Verify that all issues are fixed"""

    print("\n" + "=" * 60)
    print("VERIFYING CLEANED DATA")
    print("=" * 60)

    output = Path(OUTPUT_PATH)

    resolutions = {}
    color_modes = {}
    total = 0

    for split in ['train', 'test']:
        for class_name in ['flat foot', 'normal', 'over-arch']:
            folder = output / split / class_name
            if not folder.exists():
                continue

            for img_path in folder.iterdir():
                if img_path.suffix.lower() not in ['.jpg', '.png', '.jpeg']:
                    continue

                total += 1
                img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)

                if img is not None:
                    # Resolution
                    if len(img.shape) == 2:
                        h, w = img.shape
                    else:
                        h, w = img.shape[:2]
                    res = f"{w}x{h}"
                    resolutions[res] = resolutions.get(res, 0) + 1

                    # Color mode
                    if len(img.shape) == 2:
                        mode = "Grayscale"
                    elif len(img.shape) == 3:
                        mode = f"{img.shape[2]} channels"
                    else:
                        mode = "Unknown"
                    color_modes[mode] = color_modes.get(mode, 0) + 1

    print(f"\nTotal images in cleaned dataset: {total}")

    print(f"\nResolutions:")
    for res, count in resolutions.items():
        status = "y" if res == "224x224" else "n"
        print(f"  {status} {res}: {count} images")

    print(f"\nColor modes:")
    for mode, count in color_modes.items():
        status = "y" if mode == "Grayscale" else "n"
        print(f"  {status} {mode}: {count} images")

    # Final check
    all_correct = (len(resolutions) == 1 and "224x224" in resolutions and
                   len(color_modes) == 1 and "Grayscale" in color_modes)

    if all_correct:
        print("\n" + "=" * 60)
        print("ALL ISSUES FIXED! Dataset is clean and ready!")
        print("=" * 60)
    else:
        print("\nSome issues remain - check output above")


def report_class_distribution():
    """Report class distribution after cleaning"""

    print("\n" + "=" * 60)
    print("CLASS DISTRIBUTION")
    print("=" * 60)

    output = Path(OUTPUT_PATH)

    for split in ['train', 'test']:
        print(f"\n{split.upper()}:")
        total = 0
        counts = {}

        for class_name in ['flat foot', 'normal', 'over-arch']:
            folder = output / split / class_name
            if folder.exists():
                images = list(folder.glob('*.jpg'))
                count = len(images)
                counts[class_name] = count
                total += count

        for class_name, count in counts.items():
            percentage = (count / total * 100) if total > 0 else 0
            print(f"  {class_name}: {count} ({percentage:.1f}%)")

    print("\nNote: Class imbalance detected")
    print("   'normal' class is overrepresented")
    print("   Consider augmentation for 'flat foot' class during preprocessing")

if __name__ == "__main__":
    print("=" * 60)
    print("PLANTAR DATA CLEANING - FIX ISSUES")
    print("=" * 60)
    print(f"\nInput: {PLANTAR_PATH}")
    print(f"Output: {OUTPUT_PATH}")

    response = input("\nThis will create a new cleaned dataset. Continue? (yes/no): ")
    if response.lower() != 'yes':
        print("Cancelled.")
        exit()

    # Run cleaning
    clean_plantar_data()

    # Verify results
    verify_cleaned_data()

    # Report distribution
    report_class_distribution()

    print("\nDone! Use the cleaned dataset at:", OUTPUT_PATH)
    print("   Proceed to preprocessing next")

PLANTAR DATA CLEANING - FIX ISSUES

Input: /content/Plantar
Output: /content/Plantar_C

This will create a new cleaned dataset. Continue? (yes/no): yes
Creating output directory structure...
Output directory created: /content/Plantar_C

Processing images...

Processing TRAIN set...


  over-arch: 100%|██████████| 487/487 [00:01<00:00, 399.28it/s]


Processing TEST set...


  over-arch: 100%|██████████| 125/125 [00:00<00:00, 377.24it/s]



CLEANING COMPLETE!

Statistics:
  Total images processed: 1733
  Converted to grayscale: 1733
  Resized to 224x224: 1733
  Already correct: 0

Cleaned data saved to: /content/Plantar_C

VERIFYING CLEANED DATA

Total images in cleaned dataset: 1733

Resolutions:
  y 224x224: 1733 images

Color modes:
  y Grayscale: 1733 images

ALL ISSUES FIXED! Dataset is clean and ready!

CLASS DISTRIBUTION

TRAIN:
  flat foot: 225 (16.2%)
  normal: 680 (48.9%)
  over-arch: 487 (35.0%)

TEST:
  flat foot: 48 (14.1%)
  normal: 168 (49.3%)
  over-arch: 125 (36.7%)

Note: Class imbalance detected
   'normal' class is overrepresented
   Consider augmentation for 'flat foot' class during preprocessing

Done! Use the cleaned dataset at: /content/Plantar_C
   Proceed to preprocessing next


PREPROCESSING

In [16]:
import os
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
import scipy.ndimage as ndimage

PLANTAR_CLEANED_PATH = "/content/Plantar_C"
PLANTAR_PREPROCESSED_PATH = "/content/Plantar_P1"

# Augmentation factors per class (to balance classes)
AUGMENTATION_FACTORS = {
    'flat foot': 8,   # 225 → 1,800 images
    'normal': 2,      # 680 → 1,360 images
    'over-arch': 3    # 487 → 1,461 images
}

SAVE_AS_NPY = False

def normalize_pressure(img, body_weight=70):

    # Convert to float
    img_float = img.astype(np.float32)

    # Normalize by body weight (simulated - divide by weight factor)
    # In real scenario, you'd have actual body weight data
    # Here we just normalize by a standard weight factor
    weight_normalized = img_float / body_weight

    # Min-max normalization per image
    img_min = weight_normalized.min()
    img_max = weight_normalized.max()

    if img_max == img_min:
        return np.zeros_like(img_float)

    normalized = (weight_normalized - img_min) / (img_max - img_min)

    return normalized


def noise_filtering(img):

    # Convert to uint8 for median filter
    img_uint8 = (img * 255).astype(np.uint8)

    # Apply median filter (kernel size 5x5)
    filtered = cv2.medianBlur(img_uint8, 5)

    # Convert back to float [0, 1]
    filtered_float = filtered.astype(np.float32) / 255.0

    return filtered_float


def foot_alignment(img):

    # Convert to uint8 for contour detection
    img_uint8 = (img * 255).astype(np.uint8)

    # Threshold to get binary image
    _, binary = cv2.threshold(img_uint8, 30, 255, cv2.THRESH_BINARY)

    # Find contours
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if len(contours) == 0:
        return img

    # Get largest contour (foot)
    largest_contour = max(contours, key=cv2.contourArea)

    # Calculate moments
    moments = cv2.moments(largest_contour)

    if moments['m00'] == 0:
        return img

    # Calculate orientation angle
    # Using central moments to find principal axis
    mu20 = moments['mu20'] / moments['m00']
    mu02 = moments['mu02'] / moments['m00']
    mu11 = moments['mu11'] / moments['m00']

    # Calculate angle
    angle = 0.5 * np.arctan2(2 * mu11, mu20 - mu02)
    angle_degrees = np.degrees(angle)

    # Only rotate if angle is significant (> 5 degrees)
    if abs(angle_degrees) > 5:
        h, w = img.shape
        center = (w // 2, h // 2)

        # Get rotation matrix
        M = cv2.getRotationMatrix2D(center, angle_degrees, 1.0)

        # Rotate image
        aligned = cv2.warpAffine(img, M, (w, h),
                                borderMode=cv2.BORDER_CONSTANT,
                                borderValue=0)
        return aligned

    return img


def pressure_map_generation(img):

    # Apply Gaussian blur for smooth pressure distribution
    # Sigma = 2.0 for moderate smoothing
    smoothed = ndimage.gaussian_filter(img, sigma=2.0)

    return smoothed


def augmentation(img, class_name):

    augmented = []

    # Get augmentation factor for this class
    aug_factor = AUGMENTATION_FACTORS.get(class_name, 2)

    # Original
    augmented.append(('original', img))

    h, w = img.shape
    center = (w // 2, h // 2)

    # Calculate how many augmented versions we need
    num_augmentations = aug_factor - 1  # Subtract original

    aug_count = 0

    # 1. Horizontal flip (if needed)
    if aug_count < num_augmentations:
        flipped = cv2.flip(img, 1)
        augmented.append(('flip', flipped))
        aug_count += 1

    # 2. Rotations
    rotation_angles = [-15, -10, -5, 5, 10, 15]
    for angle in rotation_angles:
        if aug_count >= num_augmentations:
            break
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated = cv2.warpAffine(img, M, (w, h),
                                borderMode=cv2.BORDER_REFLECT)
        augmented.append((f'rot{angle}', rotated))
        aug_count += 1

    # 3. Scaling
    scale_factors = [0.9, 0.95, 1.05, 1.1]
    for scale in scale_factors:
        if aug_count >= num_augmentations:
            break

        new_h, new_w = int(h * scale), int(w * scale)
        scaled = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

        if scale > 1.0:  # Zoom in - crop center
            start_h = (new_h - h) // 2
            start_w = (new_w - w) // 2
            scaled = scaled[start_h:start_h+h, start_w:start_w+w]
        else:  # Zoom out - pad
            pad_h = (h - new_h) // 2
            pad_w = (w - new_w) // 2
            scaled = cv2.copyMakeBorder(scaled, pad_h, h-new_h-pad_h,
                                       pad_w, w-new_w-pad_w,
                                       cv2.BORDER_REFLECT)

        augmented.append((f'scale{scale}', scaled))
        aug_count += 1

    # 4. Brightness adjustments (for remaining augmentations)
    brightness_factors = [0.8, 0.9, 1.1, 1.2]
    for brightness in brightness_factors:
        if aug_count >= num_augmentations:
            break

        adjusted = img * brightness
        adjusted = np.clip(adjusted, 0, 1)  # Keep in [0, 1] range
        augmented.append((f'bright{brightness}', adjusted))
        aug_count += 1

    return augmented


def preprocess_image(img, class_name, apply_augment=False):


    # Step 1: Normalization
    normalized = normalize_pressure(img)

    # Step 2: Noise filtering
    filtered = noise_filtering(normalized)

    # Step 3: Foot alignment
    aligned = foot_alignment(filtered)

    # Step 4: Pressure map generation
    smoothed = pressure_map_generation(aligned)

    # Step 5: Augmentation (if training)
    if apply_augment:
        return augmentation(smoothed, class_name)
    else:
        return smoothed

def process_dataset():
    """Process all plantar images"""

    input_path = Path(PLANTAR_CLEANED_PATH)
    output_path = Path(PLANTAR_PREPROCESSED_PATH)

    # Create output directory
    print("Creating output directory structure...")
    for split in ['train', 'test']:
        for class_name in ['flat foot', 'normal', 'over-arch']:
            (output_path / split / class_name).mkdir(parents=True, exist_ok=True)

    print(f"Output directory: {output_path}\n")

    # Statistics
    stats = {
        'total_processed': 0,
        'train_original': 0,
        'train_augmented': 0,
        'test_processed': 0,
        'per_class': {
            'flat foot': {'original': 0, 'augmented': 0},
            'normal': {'original': 0, 'augmented': 0},
            'over-arch': {'original': 0, 'augmented': 0}
        }
    }

    # Process each split
    for split in ['train', 'test']:
        print(f"\nProcessing {split.upper()} set...")

        apply_augment = (split == 'train')

        for class_name in ['flat foot', 'normal', 'over-arch']:
            input_folder = input_path / split / class_name
            output_folder = output_path / split / class_name

            if not input_folder.exists():
                print(f"{class_name} folder not found, skipping...")
                continue

            images = list(input_folder.glob('*.jpg'))
            print(f"  Processing {class_name}: {len(images)} images (aug factor: {AUGMENTATION_FACTORS.get(class_name, 1)}x)...")

            for img_path in tqdm(images, desc=f"    {class_name}"):
                # Read image
                img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"Could not read: {img_path.name}")
                    continue

                # Preprocess
                if apply_augment:
                    # Get augmented versions
                    augmented_imgs = preprocess_image(img, class_name, apply_augment=True)

                    for aug_name, preprocessed in augmented_imgs:
                        # Create output filename
                        base_name = img_path.stem
                        if aug_name != 'original':
                            output_name = f"{base_name}_{aug_name}"
                        else:
                            output_name = base_name

                        # Save
                        if SAVE_AS_NPY:
                            output_file = output_folder / f"{output_name}.npy"
                            np.save(output_file, preprocessed)
                        else:
                            output_file = output_folder / f"{output_name}.jpg"
                            img_uint8 = (preprocessed * 255).astype(np.uint8)
                            cv2.imwrite(str(output_file), img_uint8)

                        stats['train_augmented'] += 1
                        stats['per_class'][class_name]['augmented'] += 1

                    stats['train_original'] += 1
                    stats['per_class'][class_name]['original'] += 1

                else:
                    # No augmentation (test set)
                    preprocessed = preprocess_image(img, class_name, apply_augment=False)

                    # Save
                    if SAVE_AS_NPY:
                        output_file = output_folder / f"{img_path.stem}.npy"
                        np.save(output_file, preprocessed)
                    else:
                        output_file = output_folder / f"{img_path.stem}.jpg"
                        img_uint8 = (preprocessed * 255).astype(np.uint8)
                        cv2.imwrite(str(output_file), img_uint8)

                    stats['test_processed'] += 1

                stats['total_processed'] += 1

    # Print statistics
    print("\n" + "=" * 60)
    print("PREPROCESSING COMPLETE!")
    print("=" * 60)

    print(f"\nStatistics:")
    print(f"  Total original images processed: {stats['total_processed']}")

    print(f"\n  Training set:")
    print(f"    Original images: {stats['train_original']}")
    print(f"    After augmentation: {stats['train_augmented']}")
    print(f"    Augmentation factor: {stats['train_augmented'] / stats['train_original']:.1f}x average")

    print(f"\n  Per-class breakdown (train):")
    for class_name in ['flat foot', 'normal', 'over-arch']:
        orig = stats['per_class'][class_name]['original']
        aug = stats['per_class'][class_name]['augmented']
        factor = aug / orig if orig > 0 else 0
        print(f"    {class_name}: {orig} → {aug} ({factor:.1f}x)")

    print(f"\n  Test set: {stats['test_processed']} images (no augmentation)")

    file_format = '.npy' if SAVE_AS_NPY else '.jpg'
    print(f"\n  File format: {file_format}")
    print(f"  Output location: {output_path}")

    print("\nClasses are now balanced for training!")
    print("   Preprocessing complete! Ready for feature extraction.")


def verify_preprocessed():
    """Verify preprocessed data"""

    output_path = Path(PLANTAR_PREPROCESSED_PATH)

    print("\n" + "=" * 60)
    print("VERIFYING PREPROCESSED DATA")
    print("=" * 60)

    file_ext = '.npy' if SAVE_AS_NPY else '.jpg'

    for split in ['train', 'test']:
        print(f"\n{split.upper()}:")
        total = 0

        for class_name in ['flat foot', 'normal', 'over-arch']:
            folder = output_path / split / class_name
            if folder.exists():
                files = list(folder.glob(f'*{file_ext}'))
                count = len(files)
                total += count

                percentage = (count / total * 100) if total > 0 else 0
                print(f"  {class_name}: {count} files")

                # Check a sample file
                if files:
                    sample = files[0]
                    if SAVE_AS_NPY:
                        data = np.load(sample)
                    else:
                        data = cv2.imread(str(sample), cv2.IMREAD_GRAYSCALE)
                        data = data.astype(np.float32) / 255.0

                    print(f"    Sample: {sample.name}")
                    print(f"    Shape: {data.shape}")
                    print(f"    Range: [{data.min():.4f}, {data.max():.4f}]")

        # Check class balance for train
        if split == 'train':
            print(f"\n  Class balance check:")
            for class_name in ['flat foot', 'normal', 'over-arch']:
                folder = output_path / split / class_name
                if folder.exists():
                    files = list(folder.glob(f'*{file_ext}'))
                    percentage = (len(files) / total * 100) if total > 0 else 0
                    print(f"    {class_name}: {percentage:.1f}%")


if __name__ == "__main__":
    print("=" * 60)
    print("PLANTAR DATA PREPROCESSING - FULL DATASET")
    print("=" * 60)
    print(f"\nConfiguration:")
    print(f"  Input: {PLANTAR_CLEANED_PATH}")
    print(f"  Output: {PLANTAR_PREPROCESSED_PATH}")
    print(f"  Save format: {'.npy' if SAVE_AS_NPY else '.jpg'}")
    print(f"\n  Augmentation factors:")
    for class_name, factor in AUGMENTATION_FACTORS.items():
        print(f"    {class_name}: {factor}x")

    response = input("\nStart preprocessing? (yes/no): ")
    if response.lower() != 'yes':
        print("Cancelled.")
        exit()

    # Process dataset
    process_dataset()

    # Verify
    verify_preprocessed()

    print("\nAll done! Preprocessed data ready for feature extraction.")

PLANTAR DATA PREPROCESSING - FULL DATASET

Configuration:
  Input: /content/Plantar_C
  Output: /content/Plantar_P1
  Save format: .jpg

  Augmentation factors:
    flat foot: 8x
    normal: 2x
    over-arch: 3x

Start preprocessing? (yes/no): yes
Creating output directory structure...
Output directory: /content/Plantar_P1


Processing TRAIN set...
  Processing flat foot: 225 images (aug factor: 8x)...


    flat foot: 100%|██████████| 225/225 [00:01<00:00, 119.99it/s]


  Processing normal: 680 images (aug factor: 2x)...


    normal: 100%|██████████| 680/680 [00:02<00:00, 294.62it/s]


  Processing over-arch: 487 images (aug factor: 3x)...


...

  Processing normal: 168 images (aug factor: 2x)...


    normal: 100%|██████████| 168/168 [00:00<00:00, 337.90it/s]


  Processing over-arch: 125 images (aug factor: 3x)...


    over-arch: 100%|██████████| 125/125 [00:00<00:00, 334.41it/s]



PREPROCESSING COMPLETE!

Statistics:
  Total original images processed: 1733

  Training set:
    Original images: 1392
    After augmentation: 4621
    Augmentation factor: 3.3x average

  Per-class breakdown (train):
    flat foot: 225 → 1800 (8.0x)
    normal: 680 → 1360 (2.0x)
    over-arch: 487 → 1461 (3.0x)

  Test set: 341 images (no augmentation)

  File format: .jpg
  Output location: /content/Plantar_P1

Classes are now balanced for training!
   Preprocessing complete! Ready for feature extraction.

VERIFYING PREPROCESSED DATA

TRAIN:
  flat foot: 1800 files
    Sample: image_0846_png_jpg.rf.4582c59964d85db3862ae057477c4220_rot10.jpg
    Shape: (224, 224)
    Range: [0.0000, 1.0000]
  normal: 1360 files
    Sample: image_0305_png_jpg.rf.c0a5c7f383ed601ea272550602302f51.jpg
    Shape: (224, 224)
    Range: [0.0000, 1.0000]
  over-arch: 1461 files
    Sample: image_0159_jpg.rf.def1c944fbe6b7bd45fc81dadd66bce2.jpg
    Shape: (224, 224)
    Range: [0.0000, 1.0000]

  Class balan

EXTRACTION

In [17]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

PLANTAR_PREPROCESSED_PATH = "/content/Plantar_P1"
OUTPUT_CSV = "/content/plantar_features.csv"

LOAD_FROM_NPY = False

def extract_mean_pressure(img):

    # Only consider non-zero pixels (actual foot contact)
    foot_pixels = img[img > 0]

    if len(foot_pixels) == 0:
        return 0.0

    mean_pressure = np.mean(foot_pixels)
    return mean_pressure


def extract_max_pressure(img):

    max_pressure = np.max(img)
    return max_pressure


def extract_contact_area(img):

    # Define threshold for contact (pixels > 5% of max pressure)
    threshold = 0.05

    # Count pixels above threshold
    contact_pixels = np.sum(img > threshold)

    # Total possible pixels
    total_pixels = img.shape[0] * img.shape[1]

    # Contact area as percentage
    contact_area = (contact_pixels / total_pixels) * 100

    return contact_area


def extract_center_of_pressure(img):

    # Get image dimensions
    h, w = img.shape

    # Create coordinate grids
    y_coords, x_coords = np.mgrid[0:h, 0:w]

    # Calculate total pressure (sum of all pressure values)
    total_pressure = np.sum(img)

    if total_pressure == 0:
        # No pressure detected, return center of image
        return 0.5, 0.5

    # Calculate weighted average coordinates
    cop_x = np.sum(x_coords * img) / total_pressure
    cop_y = np.sum(y_coords * img) / total_pressure

    # Normalize to [0, 1] range
    cop_x_norm = cop_x / w
    cop_y_norm = cop_y / h

    return cop_x_norm, cop_y_norm


def extract_foot_regions_ratio(img):
    h, w = img.shape

    # Divide foot into 3 horizontal regions
    # Assuming foot is oriented vertically (toes at top)
    # Forefoot (top 1/3)
    # Midfoot (middle 1/3)
    # Hindfoot (bottom 1/3)

    third_h = h // 3

    forefoot = img[0:third_h, :]
    midfoot = img[third_h:2*third_h, :]
    hindfoot = img[2*third_h:, :]

    # Calculate mean pressure in each region
    forefoot_pressure = np.mean(forefoot[forefoot > 0]) if np.any(forefoot > 0) else 0
    midfoot_pressure = np.mean(midfoot[midfoot > 0]) if np.any(midfoot > 0) else 0
    hindfoot_pressure = np.mean(hindfoot[hindfoot > 0]) if np.any(hindfoot > 0) else 0

    # Calculate total
    total = forefoot_pressure + midfoot_pressure + hindfoot_pressure

    if total == 0:
        return 0.33, 0.33, 0.33

    # Calculate ratios
    forefoot_ratio = forefoot_pressure / total
    midfoot_ratio = midfoot_pressure / total
    hindfoot_ratio = hindfoot_pressure / total

    return forefoot_ratio, midfoot_ratio, hindfoot_ratio


def extract_all_features(img):
    # Extract features
    mean_pressure = extract_mean_pressure(img)
    max_pressure = extract_max_pressure(img)
    contact_area = extract_contact_area(img)
    cop_x, cop_y = extract_center_of_pressure(img)
    forefoot_ratio, midfoot_ratio, hindfoot_ratio = extract_foot_regions_ratio(img)

    features = {
        'mean_pressure': mean_pressure,
        'max_pressure': max_pressure,
        'contact_area': contact_area,
        'cop_x': cop_x,
        'cop_y': cop_y,
        'forefoot_ratio': forefoot_ratio,
        'midfoot_ratio': midfoot_ratio,
        'hindfoot_ratio': hindfoot_ratio
    }

    return features

def process_dataset():

    preprocessed_path = Path(PLANTAR_PREPROCESSED_PATH)

    # List to store all features
    all_features = []

    print("=" * 60)
    print("PLANTAR FEATURE EXTRACTION")
    print("=" * 60)
    print(f"\nInput: {preprocessed_path}")
    print(f"Output: {OUTPUT_CSV}")

    file_ext = '.npy' if LOAD_FROM_NPY else '.jpg'

    # Process each split
    for split in ['train', 'test']:
        print(f"\nProcessing {split.upper()} set...")

        for class_name in ['flat foot', 'normal', 'over-arch']:
            folder = preprocessed_path / split / class_name

            if not folder.exists():
                print(f"{class_name} folder not found, skipping...")
                continue

            # Get all files
            files = list(folder.glob(f'*{file_ext}'))
            print(f"  Processing {class_name}: {len(files)} images...")

            for file_path in tqdm(files, desc=f"    {class_name}"):
                try:
                    # Load image
                    if LOAD_FROM_NPY:
                        img = np.load(file_path)
                    else:
                        img = cv2.imread(str(file_path), cv2.IMREAD_GRAYSCALE)
                        img = img.astype(np.float32) / 255.0  # Normalize to [0, 1]

                    if img is None:
                        print(f"Could not load: {file_path.name}")
                        continue

                    # Extract features
                    features = extract_all_features(img)

                    # Add metadata
                    features['filename'] = file_path.name
                    features['split'] = split
                    features['label'] = class_name

                    all_features.append(features)

                except Exception as e:
                    print(f"Error processing {file_path.name}: {str(e)}")
                    continue

    # Convert to DataFrame
    df = pd.DataFrame(all_features)

    # Reorder columns: metadata first, then features
    column_order = [
        'filename', 'split', 'label',
        'mean_pressure', 'max_pressure', 'contact_area',
        'cop_x', 'cop_y',
        'forefoot_ratio', 'midfoot_ratio', 'hindfoot_ratio'
    ]
    df = df[column_order]

    # Save to CSV
    output_path = Path(OUTPUT_CSV)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)

    # Print statistics
    print("\n" + "=" * 60)
    print("FEATURE EXTRACTION COMPLETE!")
    print("=" * 60)
    print(f"\nStatistics:")
    print(f"  Total images processed: {len(df)}")
    print(f"  Train images: {len(df[df['split'] == 'train'])}")
    print(f"  Test images: {len(df[df['split'] == 'test'])}")

    print(f"\n  Per-class counts:")
    for class_name in ['flat foot', 'normal', 'over-arch']:
        count = len(df[df['label'] == class_name])
        print(f"    {class_name}: {count}")

    print(f"\n  Features extracted: 8")
    print(f"    Pressure: 2 (mean, max)")
    print(f"    Spatial: 3 (contact area, CoP X, CoP Y)")
    print(f"    Regional: 3 (forefoot, midfoot, hindfoot ratios)")

    print(f"\nCSV saved to: {output_path}")

    # Show sample
    print("\nSample of extracted features:")
    print(df.head())

    return df


def verify_features(csv_path):
    """Verify the extracted features"""

    print("\n" + "=" * 60)
    print("VERIFYING FEATURES")
    print("=" * 60)

    df = pd.read_csv(csv_path)

    print(f"\nDataFrame shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")

    print("\nFeature statistics:")
    feature_cols = [
        'mean_pressure', 'max_pressure', 'contact_area',
        'cop_x', 'cop_y',
        'forefoot_ratio', 'midfoot_ratio', 'hindfoot_ratio'
    ]

    for col in feature_cols:
        print(f"\n{col}:")
        print(f"  Min: {df[col].min():.4f}")
        print(f"  Max: {df[col].max():.4f}")
        print(f"  Mean: {df[col].mean():.4f}")
        print(f"  Std: {df[col].std():.4f}")

    # Check for missing values
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print("\nMissing values found:")
        print(missing[missing > 0])
    else:
        print("\nNo missing values")

    # Check for duplicates
    duplicates = df.duplicated(subset=['filename']).sum()
    if duplicates > 0:
        print(f"\n{duplicates} duplicate filenames found")
    else:
        print("\nNo duplicate filenames")

    # Check ratio sum (should be ~1.0)
    df['ratio_sum'] = df['forefoot_ratio'] + df['midfoot_ratio'] + df['hindfoot_ratio']
    ratio_check = df['ratio_sum'].mean()
    print(f"\nRegion ratio sum check: {ratio_check:.4f} (should be ~1.0)")
    if abs(ratio_check - 1.0) < 0.01:
        print("Ratios are correct")
    else:
        print("Ratios may have issues")

if __name__ == "__main__":
    print("=" * 60)
    print("PLANTAR FEATURE EXTRACTION - FULL DATASET")
    print("=" * 60)

    response = input("\n⚠️  Start feature extraction? (yes/no): ")
    if response.lower() != 'yes':
        print("Cancelled.")
        exit()

    # Extract features
    df = process_dataset()

    # Verify
    verify_features(OUTPUT_CSV)

    print("\nFeature extraction complete!")

PLANTAR FEATURE EXTRACTION - FULL DATASET

⚠️  Start feature extraction? (yes/no): yes
PLANTAR FEATURE EXTRACTION

Input: /content/Plantar_P1
Output: /content/plantar_features.csv

Processing TRAIN set...
  Processing flat foot: 1800 images...


    flat foot: 100%|██████████| 1800/1800 [00:02<00:00, 645.32it/s]


  Processing normal: 1360 images...


    normal: 100%|██████████| 1360/1360 [00:01<00:00, 882.72it/s]


  Processing over-arch: 1461 images...


...

  Processing normal: 168 images...


    normal: 100%|██████████| 168/168 [00:00<00:00, 858.64it/s]


  Processing over-arch: 125 images...


    over-arch: 100%|██████████| 125/125 [00:00<00:00, 917.01it/s]



FEATURE EXTRACTION COMPLETE!

Statistics:
  Total images processed: 4962
  Train images: 4621
  Test images: 341

  Per-class counts:
    flat foot: 1848
    normal: 1528
    over-arch: 1586

  Features extracted: 8
    Pressure: 2 (mean, max)
    Spatial: 3 (contact area, CoP X, CoP Y)
    Regional: 3 (forefoot, midfoot, hindfoot ratios)

CSV saved to: /content/plantar_features.csv

Sample of extracted features:
                                            filename  split      label  \
0  image_0846_png_jpg.rf.4582c59964d85db3862ae057...  train  flat foot   
1  image_0411_png_jpg.rf.5e1527a361e1cb53ccff3bd1...  train  flat foot   
2  image_0324_jpg.rf.8bd2173d18b1fa2f96dd962d3d73...  train  flat foot   
3  image_0369_jpg.rf.2ecf5b54ab39719506ab1974cd4c...  train  flat foot   
4  image_0308_png_jpg.rf.555b04b72ab27555ea780eb7...  train  flat foot   

   mean_pressure  max_pressure  contact_area     cop_x     cop_y  \
0       0.673812           1.0     32.326212  0.512045  0.494507   
1